In [0]:
import requests
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from pyspark.sql.functions import current_timestamp, lit, to_date, col
from delta.tables import DeltaTable

# ── Config ────────────────────────────────────────────────────────────────────
CATALOG       = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.tileset_bronze"
dbutils.widgets.text("fred_api_key", "")
FRED_API_KEY = dbutils.widgets.get("fred_api_key")
OBSERVATION_START = "2020-01-01"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
print(f"Schema ready: {BRONZE_SCHEMA}")

In [0]:
def fetch_fred_series(series_id: str, value_col_name: str) -> None:
    """
    Fetches a FRED series, parses the JSON response into a Spark DataFrame,
    and upserts it into a Bronze Delta table using MERGE.
    """
    url = (
        f"https://api.stlouisfed.org/fred/series/observations"
        f"?series_id={series_id}"
        f"&api_key={FRED_API_KEY}"
        f"&file_type=json"
        f"&observation_start={OBSERVATION_START}"
    )

    print(f"Fetching {series_id}...")
    response = requests.get(url)
    response.raise_for_status()
    observations = response.json()["observations"]

    # Parse into list of dicts, filtering out missing values (FRED uses ".")
    rows = [
        {
            "series_id":        series_id,
            "obs_date":         row["date"],
            value_col_name:     None if row["value"] == "." else float(row["value"]),
            "_ingested_at":     datetime.utcnow()
        }
        for row in observations
        if row["value"] != "."
    ]

    print(f"  Parsed {len(rows)} observations")

    # Convert to Spark DataFrame
    df = spark.createDataFrame(rows)
    df = df.withColumn("obs_date", to_date(col("obs_date"), "yyyy-MM-dd"))

    # Target table name
    table_name = f"{BRONZE_SCHEMA}.raw_{series_id.lower()}"

    # Write using MERGE if table exists, otherwise create it fresh
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        (
            delta_table.alias("tgt")
            .merge(
                df.alias("src"),
                "tgt.series_id = src.series_id AND tgt.obs_date = src.obs_date"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"  Merged into {table_name}")
    else:
        df.write.format("delta").saveAsTable(table_name)
        print(f"  Created {table_name}")

In [0]:
fetch_fred_series("MORTGAGE30US", "rate_pct")
fetch_fred_series("HOUST",        "starts_thousands")
fetch_fred_series("PERMIT",       "permits_issued")

In [0]:
print("=== Bronze Table Row Counts ===")
tables = [
    "raw_mortgage30us",
    "raw_houst",
    "raw_permit",
    "regional_sales",
]
for table in tables:
    n = spark.table(f"{BRONZE_SCHEMA}.{table}").count()
    print(f"  {table:<25} {n:>6,} rows")

print("\n=== Null Check ===")
for table, col_name in [
    ("raw_mortgage30us", "rate_pct"),
    ("raw_houst",        "starts_thousands"),
    ("raw_permit",       "permits_issued"),
]:
    nulls = spark.table(f"{BRONZE_SCHEMA}.{table}").filter(f"{col_name} IS NULL").count()
    print(f"  {table:<25} {nulls} null values in {col_name}")

print("\n=== Sample: raw_mortgage30us ===")
spark.table(f"{BRONZE_SCHEMA}.raw_mortgage30us").orderBy("obs_date").show(5)